In [19]:
import os
import time
import pandas as pd
import requests
import spotipy
from IPython.display import display
from spotipy.oauth2 import SpotifyClientCredentials
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
loaded = load_dotenv(dotenv_path=dotenv_path, override=True)
print(f"dotenv_loaded={loaded}, dotenv_path={dotenv_path}")

dotenv_loaded=True, dotenv_path=/Users/Marcy_Student/Desktop/Spotify_Project/.env


In [20]:
# Environment variables are preferred; fall back to provided credentials for this first test call.
spotify_client_id = os.getenv("SPOTIFY_CLIENT_ID")
spotify_client_secret = os.getenv("SPOTIFY_CLIENT_SECRET") 
rapidapi_key = os.getenv("RAPIDAPI_KEY")
rapidapi_host = "track-analysis.p.rapidapi.com"

auth_manager = SpotifyClientCredentials(
    client_id=spotify_client_id,
    client_secret=spotify_client_secret,
)
sp = spotipy.Spotify(auth_manager=auth_manager)

headers = {
    "x-rapidapi-key": rapidapi_key,
    "x-rapidapi-host": rapidapi_host,
    "Content-Type": "application/json",
}

print("Spotify + RapidAPI configuration loaded")

Spotify + RapidAPI configuration loaded


In [3]:
# Replace these with the Spotify track IDs you want to analyze.
track_ids = [
    "7s25THrKz86DM225dOYwnr",  # sample ID from SoundNet docs
]

track_rows = []
for track_id in track_ids:
    try:
        meta = sp.track(track_id)
        track_rows.append(
            {
                "track_id": track_id,
                "track_name": meta.get("name"),
                "artist": ", ".join(a["name"] for a in meta.get("artists", [])),
                "album": meta.get("album", {}).get("name"),
            }
        )
    except Exception as exc:
        track_rows.append(
            {
                "track_id": track_id,
                "track_name": None,
                "artist": None,
                "album": None,
                "metadata_error": str(exc),
            }
        )

tracks_df = pd.DataFrame(track_rows)
display(tracks_df)

,track_id,track_name,artist,album,metadata_error
0,7s25THrKz86DM225dOYwnr,None,None,None,name 'sp' is not defined


In [ ]:
base_url = "https://track-analysis.p.rapidapi.com/pktx/spotify"
feature_fields = [
    "key",
    "mode",
    "tempo",
    "camelot",
    "duration",
    "popularity",
    "energy",
    "danceability",
    "happiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "loudness",
]

results = []
for track_id in track_ids:
    row = {"track_id": track_id}
    try:
        response = requests.get(
            f"{base_url}/{track_id}",
            headers=headers,
            timeout=20,
        )
        row["status_code"] = response.status_code

        if response.status_code != 200:
            row["error"] = response.text[:300]
        else:
            payload = response.json()
            for field in feature_fields:
                row[field] = payload.get(field)

            # Keep a small fingerprint of additional metadata for debugging/inspection.
            row["api_track_name"] = payload.get("name") or payload.get("track")
            row["api_artist"] = payload.get("artist")
    except requests.RequestException as exc:
        row["status_code"] = None
        row["error"] = str(exc)

    results.append(row)
    time.sleep(0.15)

analysis_df = pd.DataFrame(results)
display(analysis_df)

In [4]:
merged_df = tracks_df.merge(analysis_df, on="track_id", how="left")

success_count = int((merged_df["status_code"] == 200).sum())
if "error" not in merged_df.columns:
    merged_df["error"] = None

failed_df = merged_df.loc[
    merged_df["status_code"] != 200,
    ["track_id", "status_code", "error"],
]

print(f"Successful analyses: {success_count}/{len(merged_df)}")
if not failed_df.empty:
    print("Failed track IDs:")
    display(failed_df)

final_columns = [
    "track_id",
    "track_name",
    "artist",
    "album",
    "key",
    "mode",
    "tempo",
    "camelot",
    "energy",
    "danceability",
    "happiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "loudness",
    "duration",
    "popularity",
    "status_code",
]

final_columns = [col for col in final_columns if col in merged_df.columns]
display(merged_df[final_columns])

NameError: name 'analysis_df' is not defined